a two layer, 768 neuron LSTM got us to sub-3 loss with ~18 perplexity. whilst this is *decent*, sequence generation examples showed that the LSTM network was not able to sufficiently generalise context. for example, if a male character was given (for e.g "Once upon a time, there was a boy named Lucas"), the personal pronoun following would often be wrong ("she"). this may be due to many reasons - perhaps the name 'lucas' does not come up in the training data enough to generalise that this is a male name, or that the distance between the noun and pronoun is sufficiently large enough that the LSTM does not diffuse this information through the cell state well enough.

modern llms; including ChatGPT and Claude do not (unsurprisingly) use LSTM networks. rather, they use an architecture called *transformers*. **transformers** are *not* a recurrent neural network, and instead process the entire token sequence at once. the length of that token sequence available to input is called the *context window*.

In [5]:
import sys
from pathlib import Path
cwd = Path.cwd()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "src").is_dir())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

i attempt to manually implement a transformer model (there is a pre-baked transformer model in `torch`). a transformer has two key components:
- `self-attention`: with the $Q$ (query), $K$ (key) and $V$ (value) linear layers. $Q$ represents what we are looking for, $K$ represents what the current token is and $V$ represents what info to pass on.
- `feed-forward`: gets information from the self-attention layer and feeds forward.

there is multi-head attention which breaks up the `self-attention` layer into smaller chunks, but we omit implementing this for our homebrew model. given an input tensor $X$, we define
$$
\begin{align*}
    Q &= W_QX \\
    K &= W_KX \\
    V &= W_VX
\end{align*}
$$

Then, the self attention layer passes on
$$
\text{SelfAttention} = \text{softmax}\left(\frac{QK^T}{\sqrt{n}} \right)V
$$
where $n$ is the embedding dimension. then, the original input is added back onto the self attention layer. this is called the "residual connection".
$$
X_{interim} = X + \text{SelfAttention}(X)
$$
Then this gets passed into the feed forward layer
$$
\text{FeedForward} = W_2(W_1X_{interim} + b_1) + b_2
$$
Then, the output of the transformer block is
$$
X_{output} = X+ \text{FeedForward}(\text{SelfAttention}(X))
$$

In [122]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader

plt.rcParams["font.family"] = "Courier New"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from src.data.dataset import LMDataset

cuda


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim: int):
        super().__init__()

        # [embedding_dim, embedding_dim]
        self.q_proj = nn.Linear(embedding_dim, embedding_dim)
        self.k_proj = nn.Linear(embedding_dim, embedding_dim)
        self.v_proj = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x: torch.Tensor):
        # number of batches, number of tokens, dimension embedding
        _, T, d_model = x.shape

        # get the query, key and value matrices
        # [B, T, embedding_dim] * [embedding_dim, embedding_dim]
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # first dimension is batches
        # so we transpose the last two dimensions
        scores = Q @ K.transpose(-1, -2)
        scores = scores / math.sqrt(d_model)

        # we need to create the lower triangle, such that past tokens do not know future tokens
        mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # we softmax by the tokens, such that each token's attention weights are normalised
        weights = torch.softmax(scores, dim=-1)

        return weights @ V

class FeedForward(nn.Module):
    def __init__(self, embedding_dim: int, forward_dim: int):
        super().__init__()

        # w_2 * gelu(w_1x + b_1) + b_2
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, forward_dim),
            nn.GELU(),
            nn.Linear(forward_dim, embedding_dim)
        )

    def forward(self, x: torch.tensor):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim: int, forward_dim: int):
        """
        dimension of embedding
        """
        super().__init__()

        self.lnorm_one = nn.LayerNorm(embedding_dim)
        self.self_attention = SelfAttention(embedding_dim)
        self.lnorm_two = nn.LayerNorm(embedding_dim)
        self.feed_forward = FeedForward(embedding_dim, forward_dim)

    def forward(self, x: torch.Tensor):
       x = x + self.self_attention(self.lnorm_one(x))
       x = x + self.feed_forward(self.lnorm_two(x))
       return x

the above is a fully implemented transformer block (without multi-head attention). we can then wrap the transformer block into a decoder network such that there is a final linear layer which outputs logits. we also add a positional embedding which gives context to the model on the *position* of the token in a sequence.

In [50]:
class Decoder(nn.Module):
    def __init__(self, embedding_dim: int, forward_dim: int, vocab_dim: int, max_seq_len: int, n_layers: int):
        super().__init__()

        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(vocab_dim, embedding_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embedding_dim)
        self.transformers = nn.ModuleList([TransformerBlock(embedding_dim, forward_dim) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embedding_dim)
        self.lm = nn.Linear(embedding_dim, vocab_dim)

    def forward(self, x):
        _, T = x.shape

        assert T <= self.max_seq_len

        emb = self.embedding(x)
        pos = torch.arange(T, device=x.device)
        pos_emb = self.pos_embedding(pos)

        x = emb + pos_emb

        for transformer in self.transformers:
            x = transformer(x)

        x = self.final_norm(x)
        logits = self.lm(x)
        return logits

let's try to train this model. we train on the tinystories dataset. we try going for 3 layers (to be honest, i trained with one transformer layer and found the loss stopped at about ~3.6, but when adding 3 layers dropped pretty fast. scaling clearly helps transformers much more!)

In [131]:
from transformers import PreTrainedTokenizerFast

tokeniser = PreTrainedTokenizerFast.from_pretrained('vuiseng9/bpe-10.0k-tinystories')
tr_data = LMDataset(r'C:/data/tinystories/processed/train.bin', seq_len=128)
tst_data = LMDataset(r'C:/data/tinystories/processed/test.bin', seq_len=128)

# data loader, pin memory into cpu for fast transfer to gpu
tr_loader = DataLoader(tr_data, batch_size=64, shuffle=True, pin_memory=True, num_workers=0)

In [136]:
model = Decoder(256, 768, tokeniser.vocab_size, 128, 1).to(device)
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, fused=True)

In [ ]:
from collections import deque

loss_hist = []

model.train()
total_batches = len(tr_loader)
recent_losses = deque(maxlen=500)

for batch_idx, (X_batch, y_batch) in enumerate(tr_loader, start=1):
    X_batch = X_batch.to(device, dtype=torch.long, non_blocking=True)
    y_batch = y_batch.to(device, dtype=torch.long, non_blocking=True)

    optimiser.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(X_batch)
        loss = loss_fn(logits.reshape(-1, tokeniser.vocab_size), y_batch.reshape(-1))

    loss.backward()
    optimiser.step()

    recent_losses.append(loss.detach().item())
    recent_loss = sum(recent_losses) / len(recent_losses)

    loss_hist.append({ 'epoch': epoch, 'i': batch_idx, 'loss': loss.detach().item(), 'recent_loss': recent_loss })

    if batch_idx % 1000 == 0 or batch_idx == total_batches:
        progress = 100 * batch_idx / total_batches

        print(
            f"batch {batch_idx}/{total_batches} ({progress:.1f}%) :: "
            f"recent loss: {recent_loss:.4f}",
            flush=True,
        )

epoch 1/3 :: batch 1000/56759 (1.8%) :: recent loss: 6.8684


KeyboardInterrupt: 